# 01 · Configuración y adaptador de datos — Mortalidad neonatal NCHS (M4)

| Campo | Contenido |
|---|---|
| Secuencia del framework | **01: E1 · E2 · E3** → 02: E4 · P1 → 03: E5 · E6 · E7 · P2 · P3 → 04: E8 → 05: E8 · E9 · P4 → 06: E10 · E11 |
| Qué hace | Arma las 8 cohortes de nacimiento 2016–2023 a partir de los archivos públicos *Period/Cohort Linked Birth/Infant Death* de NCHS: nacimientos del año N unidos con las muertes de los años N y N+1 de los nacidos en N |
| Etapas | E1 (gobierno y configuración) · E2 (adaptador de datos, universo y población) · E3 (etiquetas) |
| Entradas | ZIP de NCHS en `../NCHS_data/raw/` (fuera del repositorio) |
| Salidas | `data/interim/cohort_<año>.parquet` (no se versiona) y `data/outputs_01_build_cohorts.txt` |
| Tiempo y memoria | Unos 2 minutos y 3 GB por cohorte; las cohortes ya construidas se omiten |
| Siguiente | `02_count_audit_es.ipynb` |
| Versión en inglés | `01_build_cohorts.ipynb` (mismo código) |

> **Reconstruir los Parquet.** Las cohortes que ya existen en `data/interim/` se omiten. Para volver a construirlas (por ejemplo, si cambian los archivos de NCHS o `libraries/nchs_reader.py`), poner **`REBUILD = True`** en la primera celda de código y volver a correr el notebook. Después hay que reconstruir también la tabla contrato (notebook 03).

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from libraries.nchs_reader import build_cohort, COHORTS, OUT_DIR

# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "es"
import importlib
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)      # decision texts are shown in full
import warnings
# scikit-learn >= 1.8 announces that LogisticRegression(penalty=...) will be replaced by l1_ratio. The models keep
# penalty (the hyperparameters chosen in notebook 04) so that results stay exactly as reported; the notice is hidden.
warnings.filterwarnings("ignore", message=".*'penalty' was deprecated.*", category=FutureWarning)
lines_es = []
if LANG == "es":                       # progress messages (print) also in Spanish
    import builtins
    def print(*args, **kwargs):
        builtins.print(*[translate(a, "es") if isinstance(a, str) and not a.startswith("\n== ") else a for a in args], **kwargs)

def write_outputs(path, header):
    # English text output (cited by the documents) and, from the Spanish notebook, a Spanish copy (*_es.txt).
    path.write_text(header + "\n" + "\n".join(lines) + "\n", encoding="utf-8")
    if LANG == "es":
        path.with_name(path.stem + "_es.txt").write_text(translate(header, "es") + "\n" + "\n".join(lines_es) + "\n", encoding="utf-8")


REBUILD = False      # True: rebuild every cohort even if its Parquet file exists
OUT_FILE = ROOT / "data" / "outputs_01_build_cohorts.txt"

## E1 · Gobierno y configuración — fuente, acuerdo de uso, semilla, software y regla de publicación

Se fijan antes de correr cualquier análisis: la fuente de datos y su versión, el acuerdo de uso, la semilla aleatoria, las versiones del software, los conteos de referencia que deben reproducirse y la regla de publicación. La celda muestra esa configuración y verifica que estén los archivos y los paquetes.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Los 8 ZIP de NCHS y los paquetes están (ok) | Sigue a E2 | 8 de 8 |
| Falta un ZIP | Descargarlo de la página de NCHS; si la cohorte ya está en `data/interim/` no hace falta para seguir | — |
| Falta un paquete | `pip install -r requirements.txt` con el entorno virtual activado | — |

In [2]:
# Stage S1 check: the configuration every notebook of this repository uses, shown so it can be audited.
import platform, importlib
from libraries.nchs_reader import RAW_DIR
def version(module):
    try:
        return importlib.import_module(module).__version__
    except Exception:
        return "NOT INSTALLED"
zips_ok = [(RAW_DIR / f"{y + 1}PE{y}CO.zip").exists() for y in COHORTS]
rows = [("data source", "NCHS (CDC) Period/Cohort Linked Birth/Infant Death public-use files, releases 2017/2016 to 2024/2023", "fixed"),
        ("raw files", f"{sum(zips_ok)} of {len(COHORTS)} ZIP files in {RAW_DIR}",
         "ok" if all(zips_ok) else "MISSING: download from https://www.cdc.gov/nchs/data_access/vitalstatsonline.htm"),
        ("data use", "NCHS data use agreement: statistical analysis only; no re-identification; no linkage to identifiable data; data not redistributed", "rule"),
        ("publication rule", "only aggregate outputs (data/outputs_*.txt); counts 1-9 shown as <10; record-level files stay in data/ (not versioned)", "rule"),
        ("random seed", "20261123 (notebooks 03 to 06)", "fixed"),
        ("reference counts", "written by this notebook (data/outputs_01_build_cohorts.txt) and reproduced by notebook 02 (gate G1)", "fixed"),
        ("python", platform.python_version(), "info")]
rows += [(m, version(m), "ok" if version(m) != "NOT INSTALLED" else "INSTALL (requirements.txt)")
         for m in ["pandas", "numpy", "pyarrow", "sklearn", "xgboost", "matplotlib"]]
config = pd.DataFrame(rows, columns=["item", "value", "status"]).set_index("item")
translate(config, LANG)

,valor,estado
ítem,,
fuente de datos,"Archivos de uso público NCHS (CDC) Period/Cohort Linked Birth/Infant Death, versiones 2017/2016 a 2024/2023",fijo
archivos originales,8 of 8 ZIP files in D:\Emmanuel\Uninorte\Tesis\NCHS_data\raw,ok
uso de los datos,Acuerdo de uso de datos del NCHS: solo análisis estadístico; sin reidentificación; sin enlace con datos identificables; los datos no se redistribuyen,regla
regla de publicación,solo salidas agregadas (data/outputs_*.txt); conteos 1-9 se muestran como <10; los archivos por registro quedan en data/ (no versionados),regla
semilla aleatoria,20261123 (notebooks 03 a 06),fijo
conteos de referencia,escritos por este notebook (data/outputs_01_build_cohorts.txt) y reproducidos por el notebook 02 (puerta P1),fijo
python,3.11.9,información
pandas,3.1.1,ok
numpy,2.4.6,ok


## E2 · Universo y población — construir las cohortes de nacimiento

Lee cada archivo directamente del ZIP, une los nacimientos con las muertes por número de secuencia de cohorte y año de muerte, y escribe un Parquet por cohorte con los códigos originales y las etiquetas.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Al menos el 99% de las muertes se une a su nacimiento | Sigue: se escribe la cohorte | 100% en las 8 cohortes |
| Menos del 99% | El adaptador se detiene con un error: revisar las claves de unión | — |
| Cohorte ya construida en una corrida anterior | Se omite; la tasa de unión se muestra en la sección siguiente | — |

In [3]:
summaries = []
for y in COHORTS:
    if REBUILD or not (OUT_DIR / f"cohort_{y}.parquet").exists():
        summaries.append(build_cohort(y))          # about 2 minutes and 3 GB of memory per cohort
    else:
        print(f"cohort {y}: already built")

cohorte 2016: ya construida
cohorte 2017: ya construida
cohorte 2018: ya construida
cohorte 2019: ya construida
cohorte 2020: ya construida
cohorte 2021: ya construida
cohorte 2022: ya construida
cohorte 2023: ya construida


## E2 · Universo y población — verificación de la unión de las muertes con su nacimiento

Para cada cohorte cuenta las muertes infantiles de madres residentes en EE. UU. en los archivos de defunciones y cuántas quedaron unidas a su nacimiento en el Parquet. La celda se detiene (`assert`) si alguna cohorte queda por debajo del 99%.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| % unido ≥ 99% en todas las cohortes | Pasa: las etiquetas de muerte son completas | 100% en las 8 cohortes |
| < 99% en alguna cohorte | Se detiene: revisar las claves de unión (número de secuencia de cohorte y año de muerte) | — |

In [4]:
# Linkage check: share of each cohort's infant deaths joined to their birth. build_cohort() already stops
# with an error below 99%; this cell shows the rate for every cohort, also for cohorts built in an earlier
# run, by reading only the death (numerator) files of the ZIP (a few seconds per cohort).
import zipfile
import numpy as np
from libraries.nchs_reader import cohort_members, _read_member, DEATH_FIELDS
link = []
for y in COHORTS:
    zpath, _, nums = cohort_members(y)
    with zipfile.ZipFile(zpath) as zf:
        deaths = pd.concat([_read_member(zf, m, DEATH_FIELDS) for m in nums], ignore_index=True)
    deaths = deaths[(deaths["DOB_YY"] == y) & (deaths["RESTATUS"] < 4)].dropna(subset=["CO_SEQNUM", "CO_YOD"])
    linked = int(pd.read_parquet(OUT_DIR / f"cohort_{y}.parquet", columns=["died_infant"])["died_infant"].sum())
    link.append({"cohort": y, "infant deaths of the cohort (US residents)": len(deaths),
                 "joined to their birth": linked, "% joined": round(100 * linked / len(deaths), 2)})
link = pd.DataFrame(link).set_index("cohort")
link["check (>= 99%)"] = np.where(link["% joined"] >= 99, "passes", "FAILS")
assert (link["% joined"] >= 99).all(), "fewer than 99% of the deaths joined their birth: review the linkage keys"
translate(link, LANG)

,muertes infantiles de la cohorte (residentes en EE. UU.),enlazadas con su nacimiento,% enlazadas,verificación (>= 99%)
cohorte,,,,
2016,22893,22893,100.0,cumple
2017,22167,22167,100.0,cumple
2018,21346,21346,100.0,cumple
2019,20639,20639,100.0,cumple
2020,19346,19346,100.0,cumple
2021,19965,19965,100.0,cumple
2022,20268,20268,100.0,cumple
2023,19743,19743,100.0,cumple


## E3 · Definición del desenlace — conteos de referencia por cohorte

Población: nacidos vivos de madres residentes en EE. UU. con 22 a 47 semanas. Etiqueta principal: muerte de 0 a 27 días; también se cuentan las etiquetas de sensibilidad.

| Resultado | Qué se hace | En este modelo |
|---|---|---|
| Conteos escritos en `outputs_01_build_cohorts.txt` | Son los conteos de referencia de la puerta P1: los notebooks siguientes los reproducen (`EXPECTED`) y se detienen si no coinciden | 29.829.867 nacidos vivos; 82.559 muertes neonatales |

In [5]:
rows = []
for y in COHORTS:
    d = pd.read_parquet(OUT_DIR / f"cohort_{y}.parquet",
                        columns=["RESTATUS", "COMBGEST", "CO_SEQNUM", "died_infant", "label_main",
                                 "label_early_neonatal", "died_first_24h", "label_day1_27"])
    us = d["RESTATUS"] < 4
    ga = d["COMBGEST"]
    pop = us & ga.between(22, 47).fillna(False)
    p28 = us & ga.between(28, 47).fillna(False)
    rows.append({
        "cohort": y, "births": len(d), "births_us_resident": int(us.sum()),
        "infant_deaths_linked": int(d["died_infant"].sum()),
        "ga_unknown": int((us & (ga == 99)).sum()), "ga_lt22": int((us & (ga < 22)).sum()),
        "pop_ge22wk": int(pop.sum()),
        "events_main_pop": int(d.loc[pop, "label_main"].sum()),
        "events_first24h_pop": int(d.loc[pop, "died_first_24h"].sum()),
        "events_day1_27_pop": int(d.loc[pop, "label_day1_27"].sum()),
        "events_early_neonatal_pop": int(d.loc[pop, "label_early_neonatal"].sum()),
        "pop_ge28wk": int(p28.sum()), "events_main_ge28": int(d.loc[p28, "label_main"].sum()),
    })
t = pd.DataFrame(rows).set_index("cohort").T
lines = ["Data adapter: birth cohorts built from the NCHS Period/Cohort Linked Birth/Infant Death "
         "public-use files (aggregate counts only)",
         "pop = live births to US-resident mothers with combined gestation 22-47 weeks; "
         "main label = neonatal death at 0-27 days", ""] + t.to_string().splitlines() + \
        ["", "Linkage of infant deaths to their birth (the adapter stops below 99%)"] + link.to_string().splitlines()
OUT_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")
if LANG == "es":
    lines_es = [translate(x, "es") for x in lines[:3]] + translate(t, "es").to_string().splitlines() + \
               ["", translate("Linkage of infant deaths to their birth (the adapter stops below 99%)", "es")] + translate(link, "es").to_string().splitlines()
    OUT_FILE.with_name(OUT_FILE.stem + "_es.txt").write_text("\n".join(lines_es) + "\n", encoding="utf-8")
translate(t, LANG)

cohorte,2016,2017,2018,2019,2020,2021,2022,2023
nacimientos,3956112,3864754,3801534,3757582,3619826,3669928,3676029,3605081
nacimientos_residentes_EEUU,3945875,3855500,3791712,3747540,3613647,3664292,3667758,3596017
muertes_infantiles_enlazadas,22893,22167,21346,20639,19346,19965,20268,19743
EG_desconocida,3457,2760,2509,2407,2478,2993,2671,2642
EG_menor_22,4130,4182,3960,3946,3690,3537,3563,3549
pob_22sem_o_más,3938288,3848558,3785243,3741187,3607479,3657762,3661524,3589826
eventos_principal_pob,11497,11057,10703,10249,9574,9633,9967,9879
eventos_primeras_24h_pob,5107,5356,5161,5136,4823,4556,4697,4620
eventos_día1_27_pob,6390,5701,5542,5113,4751,5077,5270,5259
eventos_neonatal_precoz_pob,8605,8222,7998,7605,7110,6988,7148,7089


## Resumen del notebook

Totales de las 8 cohortes.

In [6]:
# Summary of notebook 01
summary = pd.Series({
    "cohorts": f"{min(COHORTS)}-{max(COHORTS)}",
    "infant deaths joined to their birth (lowest cohort, %)": float(link["% joined"].min()),
    "population (US residents, 22-47 weeks)": int(t.loc["pop_ge22wk"].sum()),
    "neonatal deaths 0-27 days": int(t.loc["events_main_pop"].sum()),
    "deaths per 1,000 live births": round(1000 * t.loc["events_main_pop"].sum() / t.loc["pop_ge22wk"].sum(), 2),
    "early neonatal deaths 0-6 days": int(t.loc["events_early_neonatal_pop"].sum()),
    "neonatal deaths at 28 weeks or more": int(t.loc["events_main_ge28"].sum())}, name="value")
translate(summary.to_frame(), LANG)

,valor
cohortes,2016-2023
"muertes infantiles enlazadas con su nacimiento (cohorte más baja, %)",100.0
"población (residentes en EE. UU., 22-47 semanas)",29829867
muertes neonatales 0-27 días,82559
"muertes por 1,000 nacidos vivos",2.77
muertes neonatales precoces 0-6 días,60765
muertes neonatales a las 28 semanas o más,44097
